In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.applications import MobileNetV2
from keras import Sequential
from keras.layers import Dense,BatchNormalization,Dropout,GlobalAveragePooling2D
from keras.regularizers import L2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
test_path = r"C:\Users\user\OneDrive - REVA University\Projects\PlantHealth_AI\PestDataset\test"
train_path = r"C:\Users\user\OneDrive - REVA University\Projects\PlantHealth_AI\PestDataset\train"
validation_path = r"C:\Users\user\OneDrive - REVA University\Projects\PlantHealth_AI\PestDataset\Validation"

In [3]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 30

In [4]:
train_datagen = ImageDataGenerator(rescale=1/255.0)
val_datagen = ImageDataGenerator(rescale=1/255.0)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    validation_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

Found 45095 images belonging to 102 classes.
Found 7508 images belonging to 102 classes.


In [5]:
base_model = MobileNetV2(input_shape=(224,224,3),include_top=False,weights="imagenet")
base_model.trainable = False

In [6]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(64,activation="relu"),
    Dense(train_generator.num_classes, activation="softmax")
])

In [7]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [8]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

In [9]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

os.makedirs("backend/Models", exist_ok=True)

model.save("backend/Models/Pest_disease_model.keras")

Epoch 1/30
403/705 ━━━━━━━━━━━━━━━━━━━━ 5:55 1s/step - accuracy: 0.2108 - loss: 3.6420

KeyboardInterrupt: 

In [ ]:
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(224, 224),
    batch_size=32,
    label_mode="int",
    shuffle=False
)

In [ ]:
class_names = test_dataset.class_names

In [ ]:
test_dataset = test_dataset.map(
    lambda x, y: (tf.cast(x, tf.float32) / 255.0, y)
)

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4%}")

In [ ]:
y_true = []
y_pred = []

for images, labels in test_dataset:
    predictions = model.predict(images, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

print(cm)

In [ ]:
from collections import Counter

class_counts = Counter(train_path.targets)

print("Images in each class:\n")

for class_index, class_name in enumerate(train_path.classes):
    print(f"{class_name}: {class_counts[class_index]}")

In [ ]:
print("Number of classes:", train_generator.num_classes)

print("\nClasses:")
for class_name, class_index in train_generator.class_indices.items():
    print(class_index, ":", class_name)

In [ ]:
import numpy as np

class_counts = np.bincount(
    train_generator.classes,
    minlength=train_generator.num_classes
)

print("Images in each training class:\n")

for class_name, class_index in train_generator.class_indices.items():
    print(
        f"{class_index:3d} | "
        f"{class_name:50s} | "
        f"{class_counts[class_index]} images"
    )